# DEMO DIVRS — Khử thiên lệch (ML-10M)

So sánh **DIVRS** với baseline (MF nếu đã train trên Drive, không thì Most-Popular).
Metric dùng **đúng hàm `metrics.Metrics` của code** + eval toàn bộ user → khớp paper. **KHÔNG cần GPU**.

## 1. Setup: code + data + Drive

In [ ]:
!pip install -q faiss-cpu gradio matplotlib absl-py
import os, glob, re
REPO='/content/DIVRS'
if os.path.exists(REPO):
    !cd {REPO} && git fetch -q origin && git reset --hard -q origin/main
else:
    !git clone -q https://github.com/HatDuaa/DIVRS-reproduce.git {REPO}
%cd {REPO}
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ml10m/output/train_coo_record.npz'):
    !cd data && wget -q https://raw.githubusercontent.com/tsinghua-fib-lab/DICE/main/data/ml10m.zip && unzip -oq ml10m.zip
from google.colab import drive; drive.mount('/content/drive')
OUT_DIR='/content/drive/MyDrive/Colab Notebooks/DIVRS_output'

## 2. Nạp embedding DIVRS + baseline (MF nếu có, không thì Most-Popular)

In [ ]:
import numpy as np, scipy.sparse as sp, torch
D='data/ml10m/output/'
train=sp.load_npz(D+'train_coo_record.npz').tocsr()
test =sp.load_npz(D+'test_coo_record.npz').tocsr()
pop  =np.load(D+'popularity.npy').astype(float)
n_user,n_item=train.shape
pop_pct=pop.argsort().argsort()/(len(pop)-1)

def best_ckpt(run):
    cks=glob.glob(run+'ckpt/epoch_*.pth')
    best_ep,best_rc=None,-1
    for lf in glob.glob(run+'log/*'):
        try: txt=open(lf,encoding='utf-8',errors='ignore').read()
        except Exception: continue
        for m in re.finditer(r"VALIDATION epoch: (\d+).*?'recall': np\.float64\(([\d.]+)\)", txt):
            ep,rc=int(m.group(1)),float(m.group(2))
            if rc>best_rc: best_rc,best_ep=rc,ep
    if best_ep is not None and os.path.exists(run+f'ckpt/epoch_{best_ep}.pth'):
        return run+f'ckpt/epoch_{best_ep}.pth', best_ep, round(best_rc,4)
    mx=max(cks,key=lambda x:int(re.findall(r'epoch_(\d+)',x)[0]))
    return mx, int(re.findall(r'epoch_(\d+)',mx)[0]), None

div_runs=[r for r in glob.glob(OUT_DIR+'/*/') if 'divrs' in r.lower() and glob.glob(r+'ckpt/epoch_*.pth')]
assert div_runs, 'Khong thay run DIVRS nao co checkpoint tren Drive!'
div_run=max(div_runs, key=lambda r: len(glob.glob(r+'ckpt/epoch_*.pth')))
sd=torch.load(best_ckpt(div_run)[0],map_location='cpu')
Ud=sd['users_iv'].float().numpy(); Id=sd['items_iv'].float().numpy()
def score_divrs(u): return Ud[u] @ Id.T
print('DIVRS run:',os.path.basename(div_run.rstrip('/')))

mf_runs=[r for r in glob.glob(OUT_DIR+'/*/') if 'divrs' not in r.lower() and glob.glob(r+'ckpt/epoch_*.pth')]
BASE_NAME=None
if mf_runs:
    mf_run=max(mf_runs,key=lambda r:len(glob.glob(r+'ckpt/epoch_*.pth')))
    sdm=torch.load(best_ckpt(mf_run)[0],map_location='cpu')
    if 'users' in sdm:
        Umf=sdm['users'].float().numpy(); Imf=sdm['items'].float().numpy()
        def score_base(u): return Umf[u]@Imf.T
        BASE_NAME='MF (baseline hoc that)'; print('Baseline = MF:',os.path.basename(mf_run.rstrip('/')))
if BASE_NAME is None:
    def score_base(u): return pop
    BASE_NAME='Most-Popular (baseline)'; print('Baseline = Most-Popular (chua co MF)')

MODELS={BASE_NAME:score_base, 'DIVRS (debiased)':score_divrs}
print('embedding DIVRS:',Ud.shape,'| users',n_user,'items',n_item)

## A. Bảng so sánh — dùng đúng `metrics.Metrics` của code, eval TOÀN BỘ user (khớp paper)
Chạy ~1-2 phút (duyệt hết user). Số khớp `eval_checkpoint.ipynb` / paper.

In [ ]:
import pandas as pd
from metrics import Metrics   # dung dung ham metric goc cua code

def evaluate(score_fn,k=20,n_eval=None):
    users=np.where(np.diff(test.indptr)>0)[0]
    if n_eval: users=users[:n_eval]
    R=H=N=P=0.0; cnt=0
    for u in users:
        s=np.array(score_fn(u),dtype=float); s[train[u].indices]=-1e9
        top=np.argsort(-s)[:k]
        tp=test[u].indices; ntp=len(tp)
        if ntp==0: continue
        R+=Metrics.recall(top, test_pos=tp, num_test_pos=ntp)
        H+=Metrics.hr(top, test_pos=tp)
        N+=Metrics.ndcg(top, test_pos=tp, num_test_pos=ntp)
        P+=pop_pct[top].mean(); cnt+=1
    return R/cnt, H/cnt, N/cnt, P/cnt

rows=[]
for name,fn in MODELS.items():
    r,h,n,p=evaluate(fn,k=20)
    rows.append([name, round(r,4), round(h,4), round(n,4), f'{p*100:.1f}%'])
df=pd.DataFrame(rows, columns=['Model','Recall@20','HR@20','NDCG@20','AvgPop%'])
print(df.to_string(index=False))
print('\nPaper DIVRS (MF, ML-10M) @20:  Recall 0.1691 | HR 0.5302 | NDCG 0.1128')

## B. Đường cong: độ phổ biến TB của gợi ý theo Top-K (thấp = khử thiên lệch tốt)
Mẫu 1000 user cho nhanh (chỉ xem xu hướng độ phổ biến).

In [ ]:
import matplotlib.pyplot as plt
Ks=[5,10,20,30,40,50]
plt.figure(figsize=(6,4))
for name,fn in MODELS.items():
    y=[evaluate(fn,k=k,n_eval=1000)[3]*100 for k in Ks]
    plt.plot(Ks,y,marker='o',label=name)
plt.xlabel('Top-K'); plt.ylabel('Do pho bien TB cua goi y (%)')
plt.title('Khu thien lech: DIVRS thap hon baseline'); plt.legend(); plt.grid(alpha=.3)
plt.savefig('debias_curve.png',dpi=120,bbox_inches='tight'); plt.show()
print('Da luu debias_curve.png')

## C. WEB DEMO — slider trên, 2 ô dưới, cột kết quả ✓ trúng / − đã xem / ? mới
Hiện TẤT CẢ (không ẩn train). Top-K tới 50, ô cao cố định + cuộn trong ô. Chỉ ✓ tính vào Recall/HR; ? không phải sai.

In [ ]:
import gradio as gr
GOOD=np.argsort(-np.diff(test.indptr))[:30].tolist()
print('User ID nen thu (nhieu phim test):', GOOD[:12])
base_label, div_label = list(MODELS.keys())

def reco(uid,k):
    uid=int(uid); k=int(k)
    seen=set(train[uid].indices); watched=set(test[uid].indices); res=[]
    for name,fn in MODELS.items():
        s=np.array(fn(uid),dtype=float)
        top=np.argsort(-s)[:k]
        lines=[]
        for r,it in enumerate(top):
            if it in watched: kq='\u2713 trung (test)'
            elif it in seen:  kq='\u2212 da xem (train)'
            else:             kq='? goi y moi'
            lines.append(f'#{r+1:>3} | Item {it:<5} | pho bien:{pop_pct[it]*100:>4.0f}% | {kq}')
        nh=len(set(top)&watched); ns=len(set(top)&seen)
        head=f'>>> TRUNG(test):{nh}   da xem(train):{ns}   moi:{k-nh-ns}   | pho bien TB:{pop_pct[top].mean()*100:.0f}%'
        res.append(head+'\n'+'-'*52+'\n'+'\n'.join(lines))
    return res[0],res[1]

CSS='.mono textarea{font-family:monospace!important;font-size:13px;line-height:1.35;white-space:pre;}'
with gr.Blocks(css=CSS, title='DIVRS Debiasing Demo') as demo:
    gr.Markdown('## DIVRS Debiasing Demo \u2014 ML-10M\n'
                '\u2713 trung (test, dap an giau)  \u00b7  \u2212 da xem (train)  \u00b7  ? goi y moi (chua kiem chung). Chi \u2713 tinh vao Recall/HR; ? KHONG phai sai.')
    with gr.Row():
        uid=gr.Slider(0,n_user-1,step=1,value=int(GOOD[0]),label='User ID (thu cac so in o tren)')
        kk =gr.Slider(5,50,step=1,value=20,label='Top-K')
    with gr.Row():
        ob=gr.Textbox(label=base_label,lines=16,max_lines=16,elem_classes='mono')
        od=gr.Textbox(label=div_label, lines=16,max_lines=16,elem_classes='mono')
    for c in (uid,kk): c.change(reco,[uid,kk],[ob,od])
    demo.load(reco,[uid,kk],[ob,od])
demo.launch(share=True)

## Ghi chú
- Bảng A dùng **đúng `metrics.Metrics`** của code + eval toàn bộ user → số khớp paper / `eval_checkpoint.ipynb`.
- Web demo hiện hết (cả train) để xem; **? ≠ sai**, chỉ là chưa kiểm chứng.
- `AvgPop%` = proxy IOU (độ phổ biến TB của list). Thấp hơn = khử thiên lệch tốt hơn.